# 1. Introduction

Ce notebook produit des prévisions rétroactives pour la saison 2026 : les prévisions qui auraient pu être faites chaque semaine, à partir des vraies start lists, pour alimenter la future page de prévisions du dashboard.

Méthode :

- une fenêtre par vendredi, du 13 février au 16 octobre 2026 : les courses du vendredi au jeudi suivant inclus ;
- les engagés sont tirés des start lists publiées sur le site des start lists, extraites une seule fois puis gardées dans un fichier local ;
- les engagés de chaque fenêtre sont prévus par un appel à `forecast_upcoming_entries`, qui entraîne un modèle par date de course sur les 12 mois précédant la veille de la course, avec les features calculées la veille : aucune course n'utilise son propre résultat ni un résultat postérieur à sa veille ;
- chaque fenêtre donne une paire de fichiers (engagés, prévisions) envoyée sur le bucket S3, horodatée le vendredi à 0 h UTC ; une fenêtre sans prévision n'est pas envoyée ; une paire envoyée peut contenir des courses sans prévision.

Limites connues : les start lists sont celles publiées aujourd'hui (elles ont pu être complétées après la date de la fenêtre), et les engagés sont associés à la base de coureurs actuelle (un coureur entré en base après la date de la fenêtre est retrouvé).

# 2. Préparation

## 2.1 Modules

Seuls les avertissements du paquet sont affichés.

In [10]:
import logging
import tempfile
import time
from pathlib import Path

import pandas as pd

from access_it.common.log import setup_logging
from access_it.config import get_settings
from access_it.etl.extract.cache import SNAPSHOTS_DIR
from access_it.etl.extract.client import make_client
from access_it.etl.extract.upcoming_startlists import extract_upcoming_startlist_pages
from access_it.etl.transform.upcoming_entries import build_upcoming_entries
from access_it.export.s3 import download_latest_snapshot
from access_it.export.snapshot import read_snapshot_zip
from access_it.export.upcoming import (
    ENTRIES_KEYS, FORECASTS_KEYS, export_upcoming_entries_and_forecasts
)
from access_it.model.upcoming import forecast_upcoming_entries

setup_logging(logging.WARNING)

## 2.2 Constantes

`START_DATE` est le premier vendredi et la première date de course extraite, `LAST_FRIDAY` le dernier vendredi. `ENTRIES_FILE` est le fichier local des engagés extraits (dossier ignoré par Git). Rien n'est envoyé sur le bucket tant que `UPLOAD` vaut `False`.

In [11]:
START_DATE = "2026-02-13"
LAST_FRIDAY = pd.Timestamp("2026-10-16")
WINDOW_DAYS = 7
ENTRIES_FILE = Path("data/retroactive_entries_2026.parquet")
UPLOAD = True

## 2.3 Chargement des données

Le dernier instantané de la base est téléchargé depuis le bucket S3, puis lu.

In [12]:
bucket = get_settings().s3_bucket
snapshot_path = download_latest_snapshot(bucket, SNAPSHOTS_DIR)
tables = read_snapshot_zip(snapshot_path)
races = tables["races"]
rankings = tables["rankings"]
clubs = tables["clubs"]
riders = tables["riders"]
affiliations = tables["affiliations"]
departements = tables["departements"]
departemental_committees = tables["departemental_committees"]
regional_committees = tables["regional_committees"]

# 3. Engagés

## 3.1 Extraction ou lecture

Si le fichier local existe, il est lu. Sinon, les start lists des courses à partir de `START_DATE` sont lues sur le site des start lists, leurs engagés sont associés aux coureurs de la base, puis la table est écrite dans le fichier local.

In [13]:
if ENTRIES_FILE.exists():
    entries = pd.read_parquet(ENTRIES_FILE)
else:
    with make_client() as client:
        pages = extract_upcoming_startlist_pages(client=client, start_date=START_DATE)
    print(f"{len(pages)} startlist pages read for the races from {START_DATE}")
    entries = build_upcoming_entries(
        pages, clubs, riders, affiliations, departements,
        departemental_committees, regional_committees
    )
    ENTRIES_FILE.parent.mkdir(exist_ok=True)
    entries.to_parquet(ENTRIES_FILE, index=False)

## 3.2 Résumé

In [14]:
print(f"{entries['race_id'].nunique()} races, {len(entries)} entries")
print(f"Race dates: {entries['race_date'].min().date()} to {entries['race_date'].max().date()}")

377 races, 21207 entries
Race dates: 2026-02-15 to 2026-10-11


# 4. Fenêtres hebdomadaires

Les engagés sont limités aux courses du premier vendredi au jeudi qui suit le dernier vendredi, puis chaque course est rattachée au vendredi qui ouvre sa fenêtre. Ce vendredi est gardé à part, pour que les fichiers envoyés aient les mêmes colonnes que ceux du workflow.

In [15]:
last_race_date = LAST_FRIDAY + pd.Timedelta(days=WINDOW_DAYS - 1)
entries = entries[
    entries["race_date"].between(pd.Timestamp(START_DATE), last_race_date)
].reset_index(drop=True)
fridays = (
    entries["race_date"]
    - pd.to_timedelta((entries["race_date"].dt.weekday - 4) % 7, unit="D")
).rename("friday")
entries.groupby(fridays).agg(races=("race_id", "nunique"), entries=("rider_id", "size"))

,races,entries
friday,,
2026-02-13,6,282
2026-02-20,7,367
2026-02-27,9,851
2026-03-06,12,769
2026-03-13,6,603
2026-03-20,7,574
2026-03-27,13,755
2026-04-03,12,755
2026-04-10,10,688


# 5. Prévision

Un appel à `forecast_upcoming_entries` par fenêtre ; une course sans prévision est absente de la table des prévisions de sa fenêtre.

In [16]:
forecasts_by_friday = {}
start = time.perf_counter()
for friday, window_entries in entries.groupby(fridays):
    forecasts_by_friday[friday] = forecast_upcoming_entries(races, rankings, window_entries)
print(f"Forecasts computed in {time.perf_counter() - start:.0f} s")

Forecasts computed in 132 s


In [17]:
summary = []
for friday, window_entries in entries.groupby(fridays):
    nb_races = window_entries["race_id"].nunique()
    nb_forecast_races = forecasts_by_friday[friday]["race_id"].nunique()
    summary.append({
        "friday": friday.date(),
        "races": nb_races,
        "forecast_races": nb_forecast_races,
        "races_without_forecast": nb_races - nb_forecast_races,
        "entries": len(window_entries),
    })
pd.DataFrame(summary)

,friday,races,forecast_races,races_without_forecast,entries
0,2026-02-13,6,6,0,282
1,2026-02-20,7,7,0,367
2,2026-02-27,9,9,0,851
3,2026-03-06,12,12,0,769
4,2026-03-13,6,6,0,603
5,2026-03-20,7,7,0,574
6,2026-03-27,13,13,0,755
7,2026-04-03,12,12,0,755
8,2026-04-10,10,10,0,688
9,2026-04-17,15,15,0,784


# 6. Envoi sur le bucket

Chaque paire est horodatée le vendredi de sa fenêtre à 0 h UTC. La fonction d'envoi n'envoie rien quand la table des prévisions est vide. Seules les clés sont affichées, sans le nom du bucket. Tant que `UPLOAD` vaut `False`, rien n'est écrit ni envoyé : seules les clés qui seraient envoyées sont affichées.

In [18]:
for friday, window_entries in entries.groupby(fridays):
    forecasts = forecasts_by_friday[friday]
    timestamp = friday.tz_localize("UTC")
    keys = f"{ENTRIES_KEYS.build_key(timestamp)}, {FORECASTS_KEYS.build_key(timestamp)}"
    if not UPLOAD:
        status = f"would be uploaded: {keys}" if len(forecasts) else (
            "no forecast, would not be uploaded"
        )
    else:
        with tempfile.TemporaryDirectory() as tmp_dir:
            entries_path = Path(tmp_dir) / "entries.parquet"
            forecasts_path = Path(tmp_dir) / "forecasts.parquet"
            window_entries.to_parquet(entries_path, index=False)
            forecasts.to_parquet(forecasts_path, index=False)
            exported = export_upcoming_entries_and_forecasts(
                entries_path, forecasts_path, bucket=bucket, timestamp=timestamp
            )
        if exported is None:
            status = "no forecast, not uploaded"
        else:
            status = f"uploaded: {keys}, {len(exported[2])} old keys deleted"
    print(f"{friday.date()}: {status}")

2026-02-13: uploaded: entries/entries_20260213T000000Z.parquet, forecasts/forecasts_20260213T000000Z.parquet, 0 old keys deleted
2026-02-20: uploaded: entries/entries_20260220T000000Z.parquet, forecasts/forecasts_20260220T000000Z.parquet, 0 old keys deleted
2026-02-27: uploaded: entries/entries_20260227T000000Z.parquet, forecasts/forecasts_20260227T000000Z.parquet, 0 old keys deleted
2026-03-06: uploaded: entries/entries_20260306T000000Z.parquet, forecasts/forecasts_20260306T000000Z.parquet, 0 old keys deleted
2026-03-13: uploaded: entries/entries_20260313T000000Z.parquet, forecasts/forecasts_20260313T000000Z.parquet, 0 old keys deleted
2026-03-20: uploaded: entries/entries_20260320T000000Z.parquet, forecasts/forecasts_20260320T000000Z.parquet, 0 old keys deleted
2026-03-27: uploaded: entries/entries_20260327T000000Z.parquet, forecasts/forecasts_20260327T000000Z.parquet, 0 old keys deleted
2026-04-03: uploaded: entries/entries_20260403T000000Z.parquet, forecasts/forecasts_20260403T0000

# 7. Conclusion

- L'extraction, faite une fois (le fichier local est relu ensuite), a lu 1 017 pages de start lists et retenu 377 courses, du 15 février au 11 octobre 2026, avec 21 207 engagés.
- Les courses se répartissent sur 35 fenêtres, du vendredi 13 février au vendredi 9 octobre ; la fenêtre du 16 octobre est vide, la dernière course ayant eu lieu le 11 octobre.
- Les 377 courses ont une prévision : aucune course sans prévision. Les 35 fenêtres sont prévues en un peu plus de 2 minutes.
- Les 35 paires sont envoyées sur le bucket, horodatées du 13 février au 9 octobre 2026 à 0 h UTC ; aucune clé n'est supprimée.

Ces paires alimentent la page de prévisions du dashboard. Pendant la saison 2027, la rotation à 100 fichiers les remplacera progressivement par les paires du workflow.